https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C01589

Nclim-grid

In [1]:
# Import necessary libraries for data processing and raster operations.
import pandas as pd  # For handling tabular data (CSV files).
import rasterio  # For reading and sampling raster data (TIFF files).
import sys  # For accessing the Python version.

# Debug: Confirm that imports are successful.
print("Debug: Libraries imported successfully.")

# Print the versions of Python and each imported module.
print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")
print(f"rasterio version: {rasterio.__version__}")

Debug: Libraries imported successfully.
Python version: 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
pandas version: 2.2.3
rasterio version: 1.4.3


In [2]:
# Define directory paths for Kaggle environment.
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets.
sub_dir = r"/kaggle/working/"  # Submission directory for output files.

# Define file paths for the TIFF files.
air_density_path = f"{base_dir}/USA_air-density_10m.tif"
power_density_path = f"{base_dir}/USA_power-density_10m.tif"
wind_speed_path = f"{base_dir}/USA_wind-speed_10m.tif"

# Define file paths for training and validation CSV datasets.
train_file = f"{base_dir}/Training_data.csv"
valid_file = f"{base_dir}/Validation_data.csv"

# Define output file paths for the updated datasets.
output_train_csv = f"{sub_dir}/Training_data_with_tif.csv"
output_valid_csv = f"{sub_dir}/Validation_data_with_tif.csv"

# Debug: Print the file paths to confirm they are set correctly.
print(f"Debug: Air density TIFF path: {air_density_path}")
print(f"Debug: Power density TIFF path: {power_density_path}")
print(f"Debug: Wind speed TIFF path: {wind_speed_path}")
print(f"Debug: Training file path: {train_file}")
print(f"Debug: Validation file path: {valid_file}")
print(f"Debug: Output training CSV path: {output_train_csv}")
print(f"Debug: Output validation CSV path: {output_valid_csv}")

Debug: Air density TIFF path: /kaggle/input/eyds-base-dataset/USA_air-density_10m.tif
Debug: Power density TIFF path: /kaggle/input/eyds-base-dataset/USA_power-density_10m.tif
Debug: Wind speed TIFF path: /kaggle/input/eyds-base-dataset/USA_wind-speed_10m.tif
Debug: Training file path: /kaggle/input/eyds-base-dataset/Training_data.csv
Debug: Validation file path: /kaggle/input/eyds-base-dataset/Validation_data.csv
Debug: Output training CSV path: /kaggle/working//Training_data_with_tif.csv
Debug: Output validation CSV path: /kaggle/working//Validation_data_with_tif.csv


In [3]:
# --- Step 1: Read the CSV datasets ---
# Load the training dataset.
print("Loading training dataset...")
try:
    training_df = pd.read_csv(train_file)
except FileNotFoundError:
    print(f"Error: Training data file not found at {train_file}")


# Debug: Print the shape and columns of the training dataset.
print(f"Debug: Training DataFrame shape: {training_df.shape}")
print(f"Debug: Training DataFrame columns: {training_df.columns.tolist()}")

# Load the validation dataset.
print("Loading validation dataset...")
try:
    validation_df = pd.read_csv(valid_file)
except FileNotFoundError:
    print(f"Error: Validation data file not found at {valid_file}")
    

# Debug: Print the shape and columns of the validation dataset.
print(f"Debug: Validation DataFrame shape: {validation_df.shape}")
print(f"Debug: Validation DataFrame columns: {validation_df.columns.tolist()}")

def extract_values(df, raster_path, lon_col="Longitude", lat_col="Latitude"):
    """
    Extracts pixel values from a raster file for each (lon, lat) in the DataFrame.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing longitude and latitude columns.
        raster_path (str): Path to the raster file (e.g., TIFF).
        lon_col (str): Name of the longitude column in the DataFrame (default: "Longitude").
        lat_col (str): Name of the latitude column in the DataFrame (default: "Latitude").
    
    Returns:
        list: List of extracted raster values corresponding to each (lon, lat) pair.
    """
    # Prepare a list of (lon, lat) tuples.
    coords = [(x, y) for x, y in zip(df[lon_col], df[lat_col])]
    print(f"\nDebug: Extracting values from {raster_path}")
    print(f"Debug: Number of coordinates: {len(coords)}")
    
    # Open the raster file and extract values.
    try:
        with rasterio.open(raster_path) as src:
            print(f"Debug: Raster info - Bands: {src.count}, CRS: {src.crs}, Bounds: {src.bounds}")
            values = []
            for i, val in enumerate(src.sample(coords)):
                if i < 5:  # Print first 5 extracted values for debugging.
                    print(f"Debug: Coord {coords[i]} -> extracted value: {val}")
                values.append(val[0])
            print(f"Debug: Total values extracted: {len(values)} from {raster_path}")
    except Exception as e:
        print(f"Error: Failed to extract values from {raster_path}. Error: {str(e)}")
        return [None] * len(coords)  # Return a list of None values if extraction fails.
    
    return values

Loading training dataset...
Debug: Training DataFrame shape: (11229, 4)
Debug: Training DataFrame columns: ['Longitude', 'Latitude', 'datetime', 'UHI Index']
Loading validation dataset...
Debug: Validation DataFrame shape: (1040, 3)
Debug: Validation DataFrame columns: ['Longitude', 'Latitude', 'UHI Index']


In [4]:
# --- Step 2: Extract raster values and add them as new columns ---
# For each dataset, extract the three raster variables (air density, power density, wind speed).
for df, name in zip([training_df, validation_df], ['Training', 'Validation']):
    print(f"\nDebug: Processing {name} dataset...")
    df["air_density"] = extract_values(df, air_density_path)
    df["power_density"] = extract_values(df, power_density_path)
    df["wind_speed"] = extract_values(df, wind_speed_path)
    
    # Debug: Print a sample of the new columns.
    print(f"Debug: Sample of {name} DataFrame with new columns:\n{df[['air_density', 'power_density', 'wind_speed']].head()}")


Debug: Processing Training dataset...

Debug: Extracting values from /kaggle/input/eyds-base-dataset/USA_air-density_10m.tif
Debug: Number of coordinates: 11229
Debug: Raster info - Bands: 1, CRS: EPSG:4326, Bounds: BoundingBox(left=-180.00092731781535, bottom=15.563268747733929, right=179.9990726822025, top=74.71076874773686)
Debug: Coord (-73.90916667, 40.81310667) -> extracted value: [1.2347891]
Debug: Coord (-73.90918667, 40.813045) -> extracted value: [1.2347891]
Debug: Coord (-73.909215, 40.81297833) -> extracted value: [1.2347891]
Debug: Coord (-73.90924167, 40.81290833) -> extracted value: [1.2347891]
Debug: Coord (-73.90925667, 40.812845) -> extracted value: [1.2347891]


/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in greater
  data = read(indexes, window=win, masked=masked)
/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in less
  data = read(indexes, window=win, masked=masked)


Debug: Total values extracted: 11229 from /kaggle/input/eyds-base-dataset/USA_air-density_10m.tif

Debug: Extracting values from /kaggle/input/eyds-base-dataset/USA_power-density_10m.tif
Debug: Number of coordinates: 11229
Debug: Raster info - Bands: 1, CRS: EPSG:4326, Bounds: BoundingBox(left=-180.00092731781535, bottom=15.563268747733929, right=179.9990726822025, top=74.71076874773686)
Debug: Coord (-73.90916667, 40.81310667) -> extracted value: [27.27473]
Debug: Coord (-73.90918667, 40.813045) -> extracted value: [27.27473]
Debug: Coord (-73.909215, 40.81297833) -> extracted value: [27.27473]
Debug: Coord (-73.90924167, 40.81290833) -> extracted value: [27.27473]
Debug: Coord (-73.90925667, 40.812845) -> extracted value: [27.27473]


/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in greater
  data = read(indexes, window=win, masked=masked)
/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in less
  data = read(indexes, window=win, masked=masked)


Debug: Total values extracted: 11229 from /kaggle/input/eyds-base-dataset/USA_power-density_10m.tif

Debug: Extracting values from /kaggle/input/eyds-base-dataset/USA_wind-speed_10m.tif
Debug: Number of coordinates: 11229
Debug: Raster info - Bands: 1, CRS: EPSG:4326, Bounds: BoundingBox(left=-180.00092731781535, bottom=15.563268747733929, right=179.9990726822025, top=74.71076874773686)
Debug: Coord (-73.90916667, 40.81310667) -> extracted value: [2.6912918]
Debug: Coord (-73.90918667, 40.813045) -> extracted value: [2.6912918]
Debug: Coord (-73.909215, 40.81297833) -> extracted value: [2.6912918]
Debug: Coord (-73.90924167, 40.81290833) -> extracted value: [2.6912918]
Debug: Coord (-73.90925667, 40.812845) -> extracted value: [2.6912918]


/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in greater
  data = read(indexes, window=win, masked=masked)
/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in less
  data = read(indexes, window=win, masked=masked)


Debug: Total values extracted: 11229 from /kaggle/input/eyds-base-dataset/USA_wind-speed_10m.tif
Debug: Sample of Training DataFrame with new columns:
   air_density  power_density  wind_speed
0     1.234789      27.274731    2.691292
1     1.234789      27.274731    2.691292
2     1.234789      27.274731    2.691292
3     1.234789      27.274731    2.691292
4     1.234789      27.274731    2.691292

Debug: Processing Validation dataset...

Debug: Extracting values from /kaggle/input/eyds-base-dataset/USA_air-density_10m.tif
Debug: Number of coordinates: 1040
Debug: Raster info - Bands: 1, CRS: EPSG:4326, Bounds: BoundingBox(left=-180.00092731781535, bottom=15.563268747733929, right=179.9990726822025, top=74.71076874773686)
Debug: Coord (-73.971665, 40.78876333) -> extracted value: [1.231396]
Debug: Coord (-73.97192833, 40.788875) -> extracted value: [1.231396]
Debug: Coord (-73.96708, 40.78908) -> extracted value: [1.2308483]
Debug: Coord (-73.97255, 40.78908167) -> extracted value: [

/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in greater
  data = read(indexes, window=win, masked=masked)
/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in less
  data = read(indexes, window=win, masked=masked)
/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in greater
  data = read(indexes, window=win, masked=masked)
/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in less
  data = read(indexes, window=win, masked=masked)


Debug: Total values extracted: 1040 from /kaggle/input/eyds-base-dataset/USA_power-density_10m.tif

Debug: Extracting values from /kaggle/input/eyds-base-dataset/USA_wind-speed_10m.tif
Debug: Number of coordinates: 1040
Debug: Raster info - Bands: 1, CRS: EPSG:4326, Bounds: BoundingBox(left=-180.00092731781535, bottom=15.563268747733929, right=179.9990726822025, top=74.71076874773686)
Debug: Coord (-73.971665, 40.78876333) -> extracted value: [2.893831]
Debug: Coord (-73.97192833, 40.788875) -> extracted value: [2.893831]
Debug: Coord (-73.96708, 40.78908) -> extracted value: [3.66011]
Debug: Coord (-73.97255, 40.78908167) -> extracted value: [2.893831]
Debug: Coord (-73.96969667, 40.78795333) -> extracted value: [3.4455934]
Debug: Total values extracted: 1040 from /kaggle/input/eyds-base-dataset/USA_wind-speed_10m.tif
Debug: Sample of Validation DataFrame with new columns:
   air_density  power_density  wind_speed
0     1.231396      33.236259    2.893831
1     1.231396      33.236259

/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in greater
  data = read(indexes, window=win, masked=masked)
/usr/local/lib/python3.10/dist-packages/rasterio/sample.py:95: RuntimeWarning: invalid value encountered in less
  data = read(indexes, window=win, masked=masked)


In [5]:
# --- Step 3: Save the updated datasets to new CSV files ---
print("\nSaving updated datasets to CSV...")
training_df.to_csv(output_train_csv, index=False)
validation_df.to_csv(output_valid_csv, index=False)

# Debug: Print the final confirmation messages with file paths.
print(f"Debug: Updated training data saved to: {output_train_csv}")
print(f"Debug: Updated validation data saved to: {output_valid_csv}")

print("\nUpdated CSV files saved:")
print(output_train_csv)
print(output_valid_csv)


Saving updated datasets to CSV...
Debug: Updated training data saved to: /kaggle/working//Training_data_with_tif.csv
Debug: Updated validation data saved to: /kaggle/working//Validation_data_with_tif.csv

Updated CSV files saved:
/kaggle/working//Training_data_with_tif.csv
/kaggle/working//Validation_data_with_tif.csv
